# Introduction

In contemporary recommender systems, effectively leveraging user feedback is essential for improving recommendation quality and enhancing user experience. User feedback can be divided into explicit and implicit categories. Explicit feedback, such as ratings and likes, provides a direct indication of user preferences, whereas implicit feedback is derived from user behaviors like clicks and purchase history.

Different strategies are required to manage these feedback types. It is important not to interpret missing values in explicit feedback as zeros, as this could misrepresent user sentiments. In contrast, filling in missing values with zeros in implicit feedback can often signify a lack of interest.

The sparsity of user ratings presents significant challenges for conventional algorithms like K-Nearest Neighbors (KNN). Therefore, addressing this sparsity is crucial for constructing effective recommendation models. While accuracy remains a primary metric for assessing recommender systems, it is also important to consider secondary metrics such as diversity—ensuring varied recommendations—and serendipity—offering unexpected yet relevant suggestions—which further enhance user engagement and satisfaction.

This project aims to develop a hybrid recommender system utilizing a meta-level approach that combines content-based and collaborative filtering techniques. This system will specifically focus on comparing the performance between Singular Value Decomposition (SVD) and the proposed meta-level hybrid recommender. By extracting movie details from IMDb, the system will group users with similar preferences based on content features and then apply collaborative filtering for making predictions.

This tailored method, referred to as "collaboration via content," seeks to improve traditional collaborative filtering by integrating content data to identify similar users. Although it proves effective for recommending movies within specific genres, this approach can also promote diversified recommendations in industries looking to encourage users to explore new products, especially during periods of business expansion.

# TL:DR

### Performance Comparison
- **Ordinary SVD with zero-filling:** RMSE = 0.6826 (better performance)
- **SGD Model:** RMSE = 0.9578 (poorer performance)

### Interpretation of Results
- SVD outperforms SGD due to its simplicity, especially with small datasets.

### Data Limitations
- Small dataset limits SGD's generalization ability; SVD remains robust in such cases.

### Model Complexity
- SGD may overfit or miss patterns due to its complexity.
- Zero-filling SVD is straightforward and provides stable results.


In [47]:
import warnings

# Suppress all warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import ast
import nltk
import re 
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.metrics import mean_squared_error

In [4]:
movies_df = pd.read_csv("/content/movies_metadata.csv")
ratings_df = pd.read_csv("/content/ratings.csv")

# Preprocessing

To handle bias in ratings, where some users might give consistently high or low ratings, I will use centered ratings. This means adjusting each user's rating by subtracting their average rating. Additionally, we will using several techinique to handle the text such as wordnet and stopwords.

In [5]:
proc_movie = movies_df[['genres', 'adult', 'original_title', 'original_language', 'overview', 'production_countries', 'production_companies', 'release_date', 'revenue', 'runtime', 'title', 'vote_average']]

In [6]:
proc_movie['genres'] = proc_movie['genres'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
proc_movie['genre_names'] = proc_movie['genres'].apply(lambda x: ', '.join([d['name'] for d in x]) if isinstance(x, list) else '')

proc_movie['production_countries'] = proc_movie['production_countries'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
proc_movie['production_countries_code'] = proc_movie['production_countries'].apply(lambda x: ', '.join([d['iso_3166_1'] for d in x]) if isinstance(x, list) else '')

proc_movie['production_companies'] = proc_movie['production_companies'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
proc_movie['production_companies_name'] = proc_movie['production_companies'].apply(lambda x: ', '.join([d['name'] for d in x]) if isinstance(x, list) else '')

proc_movie[['release_year', 'release_month', 'release_day']] = proc_movie['release_date'].str.split("-", expand=True)

In [7]:
proc_movie.drop(columns = ['genres', 'production_countries', 'production_companies', 'release_date'], inplace = True)

In [8]:
object_columns = proc_movie.select_dtypes(include='object').columns
# Apply lowercase transformation to each column
proc_movie[object_columns] = proc_movie[object_columns].apply(lambda x: x.str.lower())

In [9]:
ratings_df['centered_rating'] = ratings_df['rating'] - ratings_df.groupby('userId')['rating'].transform('mean')

## Content-based filtering

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [11]:
proc_movie['content'] = (
    proc_movie['original_title'] + ' ' + 
    proc_movie['overview'] + ' ' + 
    proc_movie['genre_names'] + ' ' +
    proc_movie['genre_names'] + ' ' +
    proc_movie['production_companies_name']
)

In [12]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /Users/top/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /Users/top/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/top/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/top/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [13]:
from nltk.corpus import stopwords
import re
import string
from nltk.stem import WordNetLemmatizer
from nltk import word_tokenize
from nltk.corpus import stopwords
stop = stopwords.words('english')
stop_words = set(stopwords.words('english'))
wn = WordNetLemmatizer()

def is_valid_token(token):
    return token not in stop_words and token not in string.punctuation and len(token) > 2   

def clean_text(text):
    if not isinstance(text, str): 
        return "" 

    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\d+", " ", text)
    text = text.lower().replace("nbsp", "")

    clean_text = [wn.lemmatize(word) for word in word_tokenize(text) if is_valid_token(word)]
    
    return " ".join(clean_text)

# Apply the cleaning function to the 'content' column
proc_movie['content'] = proc_movie['content'].apply(clean_text)

In [15]:
proc_movie['content'] = proc_movie['content'].fillna('')
tfidf_vectorizer = TfidfVectorizer(stop_words='english')

tfidf_matrix = tfidf_vectorizer.fit_transform(proc_movie['content'])
cosine_sim_tfidf = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Collaborative filtering

The structure of a rating matrix differs from traditional machine learning tasks, where you typically have clear dependent (target) and independent (feature) variables. In a rating matrix, there isn't a straightforward distinction between these two. Instead, the matrix contains user-item interactions (like movie ratings), where each entry represents a user's rating for an item. This makes it challenging to directly apply typical machine learning techniques.

To evaluate the model's performance, a common approach used in recommender systems is the hold-out method. In this method, a portion of the ratings is "hidden" or set aside as a test set, while the remaining ratings are used for training the model. The hidden ratings are then used to check how well the model can predict unseen data, allowing for a more realistic assessment of its accuracy in making recommendations.

In [16]:
class MatrixFactorizationWithBias:
    def __init__(self, R, k, alpha, lambda_reg, num_epochs, tolerance=1e-4):
        self.R = R
        self.num_users, self.num_items = R.shape
        self.k = k
        self.alpha = alpha
        self.lambda_reg = lambda_reg
        self.num_epochs = num_epochs
        self.tolerance = tolerance
        
        
        # Uniform initialization
        self.U = np.random.uniform(low=-0.01, high=0.01, size=(self.num_users, k))
        self.V = np.random.uniform(low=-0.01, high=0.01, size=(self.num_items, k))

    def train(self):
        previous_mse = float('inf')
        for epoch in range(self.num_epochs):
            self.gradient_descent()
            mse = self.compute_error()
            print(f"Epoch: {epoch + 1}, MSE: {mse:.4f}")

            if abs(previous_mse - mse) < self.tolerance:
                print("Convergence reached.")
                break
            previous_mse = mse

    def gradient_descent(self):
        S = np.argwhere(~np.isnan(self.R))
        np.random.shuffle(S)

        for i, j in S:
            prediction = self.predict(i, j)
            eij = self.R[i, j] - prediction
            
            if not np.isnan(eij):
                for q in range(self.k):
                    # Compute updates
                    update_U = self.alpha * (eij * self.V[j, q] - self.lambda_reg * self.U[i, q])
                    update_V = self.alpha * (eij * self.U[i, q] - self.lambda_reg * self.V[j, q])

                    # Update the latent factors
                    self.U[i, q] += update_U
                    self.V[j, q] += update_V

    def predict(self, i, j):
        return np.dot(self.U[i], self.V[j])

    def compute_error(self):
        xs, ys = np.argwhere(~np.isnan(self.R)).T
        predicted = np.array([self.predict(i, j) for i, j in zip(xs, ys)])
        observed_ratings = self.R[xs, ys]

        mse = np.nanmean((observed_ratings - predicted) ** 2)
        rmse = np.sqrt(mse) if not np.isnan(mse) else float('nan')
        
        return rmse


In [17]:
def create_ratings_matrix(df):
    return df.pivot(index='userId', columns='movieId', values='centered_rating')

In [18]:
def split_data_holdout_entry_based(ratings_matrix, test_fraction=0.2):
    R = ratings_matrix.values.copy()  

    observed_indices = np.array(np.where(~np.isnan(R))).T
    np.random.shuffle(observed_indices)
    
    num_test = int(len(observed_indices) * test_fraction)
    
    test_indices = observed_indices[:num_test]
    train_indices = observed_indices[num_test:]
    
    R_train = R.copy()
    R_test = np.full(R.shape, np.nan)

    for idx in test_indices:
        i, j = idx
        R_test[i, j] = R[i, j] 
        R_train[i, j] = np.nan  
    
    print("Original Data Size:", R.shape)
    print("Training Set Size:", np.sum(~np.isnan(R_train)))
    print("Test Set Size:", np.sum(~np.isnan(R_test)))
    
    return R_train, R_test, test_indices  

In [35]:
class MetaHybridRecommender:
    def __init__(self, ratings_matrix, cosine_sim, k, alpha, lambda_reg, num_epochs):
        self.ratings_matrix = ratings_matrix
        self.cosine_sim = cosine_sim
        self.cf_model = MatrixFactorizationWithBias(ratings_matrix, k, alpha, lambda_reg, num_epochs)

    def train(self, R_train, test_indices):
        for epoch in range(self.cf_model.num_epochs):
            self.cf_model.gradient_descent()
            mse, rmse = self.compute_mse_rmse(R_train, test_indices)

    def recommend_all_users(self, num_recommendations=5):
        cf_predictions = self.cf_model.predict_all_users()
        all_recommendations = []
        for user_id in range(self.ratings_matrix.shape[0]):
            content_predictions = self.get_content_based_recommendations(user_id)
            combined_predictions = self.combine_predictions(cf_predictions[user_id], content_predictions)
            recommended_indices = np.argsort(combined_predictions)[::-1][:num_recommendations]
            all_recommendations.append(recommended_indices)
        return all_recommendations

    def get_content_based_recommendations(self, user_id, similarity_type='tfidf'):
        user_ratings = self.ratings_matrix[user_id]
        rated_indices = np.argwhere(~np.isnan(user_ratings)).flatten()

        sim_scores = self.cosine_sim if similarity_type == 'tfidf' else self.cosine_sim_count
        weighted_scores = np.zeros(self.ratings_matrix.shape[1])

        for idx in rated_indices:
            weighted_scores += sim_scores[idx] * user_ratings[idx]

        weighted_scores /= len(rated_indices) if len(rated_indices) > 0 else 1
        return weighted_scores

    def combine_predictions(self, cf_scores, content_scores, alpha=0.5, beta=0.5):
        return (alpha * cf_scores) + (beta * content_scores)

    def predict_all_users(self):
        return np.array([self.cf_model.predict(i, j) for i in range(self.ratings_matrix.shape[0]) for j in range(self.ratings_matrix.shape[1])]).reshape(self.ratings_matrix.shape)

    def compute_mse_rmse(self, R_test, test_indices):
        # Only consider the entries in the test set for RMSE calculation
        test_predictions = []
        for i, j in test_indices:
            prediction = self.cf_model.predict(i, j)
            test_predictions.append(prediction)

        actuals = R_test[test_indices[:, 0], test_indices[:, 1]]
        mse = np.mean((np.array(test_predictions) - actuals) ** 2)
        rmse = np.sqrt(mse) if len(test_predictions) > 0 else float('nan')

        return mse, rmse


In [38]:
param_grid = {
    'k': [5, 10, 15],           
    'alpha': [0.00001, 0.0001, 0.001],
    'lambda_reg': [0.01, 0.1],
    'num_epochs': [10, 20]   
}
best_params = None
best_rmse = float('inf')
best_recommender = None

The relationship between k (latent factors), alpha (learning rate), and lambda_reg (regularization) is simple:

k (Latent Factors): Increasing k improves the model's accuracy by capturing more patterns, but the benefits decrease as k gets larger, making the model slower to train.

alpha (Learning Rate): A higher alpha helps the model learn faster but can cause instability at the start. A moderate alpha balances speed and stability.

lambda_reg (Regularization): Regularization prevents the model from overfitting by limiting complexity. A steady value of lambda_reg keeps the model stable across different settings of k and alpha.

Together, k controls complexity, alpha affects learning speed, and lambda_reg ensures stability. Finding the right balance leads to faster training and better predictions.

In [37]:
rating_matrix = create_ratings_matrix(ratings_df)

In [39]:
R_train, R_test, test_indices = split_data_holdout_entry_based(rating_matrix, test_fraction=0.2)

best_rmse = float('inf')
best_params = None
best_recommender = None

for params in ParameterGrid(param_grid):
    print(f"Testing parameters: {params}")

    recommender = MetaHybridRecommender(
        ratings_matrix=R_train,
        cosine_sim=cosine_sim_tfidf,
        k=params['k'], 
        alpha=params['alpha'], 
        lambda_reg=params['lambda_reg'], 
        num_epochs=params['num_epochs']
    )

    try:
        recommender.train(R_train, test_indices)
        print("Training completed successfully.")
    except Exception as e:
        print(f"Error during training: {e}")
        continue

    try:
        mse, rmse = recommender.compute_mse_rmse(R_test, test_indices)
        print(f"MSE: {mse}, RMSE: {rmse} for parameters {params}")
    except Exception as e:
        print(f"Error during RMSE computation: {e}")
        continue

    if rmse < best_rmse:
        best_rmse = rmse
        best_params = params
        best_recommender = recommender
        print(f"New best parameters found: {best_params} with RMSE: {best_rmse}")

print(f"Best parameters: {best_params}")
print(f"Best RMSE: {best_rmse}")


Original Data Size: (671, 9066)
Training Set Size: 80004
Test Set Size: 20000
Testing parameters: {'alpha': 1e-05, 'k': 5, 'lambda_reg': 0.01, 'num_epochs': 10}
Training completed successfully.
MSE: 0.9173894944335043, RMSE: 0.9578045178602491 for parameters {'alpha': 1e-05, 'k': 5, 'lambda_reg': 0.01, 'num_epochs': 10}
New best parameters found: {'alpha': 1e-05, 'k': 5, 'lambda_reg': 0.01, 'num_epochs': 10} with RMSE: 0.9578045178602491
Testing parameters: {'alpha': 1e-05, 'k': 5, 'lambda_reg': 0.01, 'num_epochs': 20}
Training completed successfully.
MSE: 0.9173888012376908, RMSE: 0.9578041559931189 for parameters {'alpha': 1e-05, 'k': 5, 'lambda_reg': 0.01, 'num_epochs': 20}
New best parameters found: {'alpha': 1e-05, 'k': 5, 'lambda_reg': 0.01, 'num_epochs': 20} with RMSE: 0.9578041559931189
Testing parameters: {'alpha': 1e-05, 'k': 5, 'lambda_reg': 0.1, 'num_epochs': 10}
Training completed successfully.
MSE: 0.9173889551235683, RMSE: 0.957804236325758 for parameters {'alpha': 1e-05

Training completed successfully.
MSE: 0.9173883029616628, RMSE: 0.9578038958793511 for parameters {'alpha': 0.001, 'k': 10, 'lambda_reg': 0.1, 'num_epochs': 20}
Testing parameters: {'alpha': 0.001, 'k': 15, 'lambda_reg': 0.01, 'num_epochs': 10}
Training completed successfully.
MSE: 0.9173881837676038, RMSE: 0.9578038336567691 for parameters {'alpha': 0.001, 'k': 15, 'lambda_reg': 0.01, 'num_epochs': 10}
Testing parameters: {'alpha': 0.001, 'k': 15, 'lambda_reg': 0.01, 'num_epochs': 20}
Training completed successfully.
MSE: 0.9173898623819868, RMSE: 0.9578047099393419 for parameters {'alpha': 0.001, 'k': 15, 'lambda_reg': 0.01, 'num_epochs': 20}
Testing parameters: {'alpha': 0.001, 'k': 15, 'lambda_reg': 0.1, 'num_epochs': 10}
Training completed successfully.
MSE: 0.9173878455920683, RMSE: 0.9578036571198025 for parameters {'alpha': 0.001, 'k': 15, 'lambda_reg': 0.1, 'num_epochs': 10}
Testing parameters: {'alpha': 0.001, 'k': 15, 'lambda_reg': 0.1, 'num_epochs': 20}
Training completed s

Best Parameters Found:

Parameters: {'alpha': 0.0001, 'k': 10, 'lambda_reg': 0.01, 'num_epochs': 10}
RMSE: 0.9578021673659756
Overall Performance:

The RMSE values fluctuate slightly across different parameter combinations, but the best-performing combination so far has been with alpha = 0.0001, k = 10, lambda_reg = 0.01, and num_epochs = 10.
The smallest RMSE recorded is approximately 0.9578.

# SVD with 0 filling

In [48]:
ratings_matrix_svd = rating_matrix.fillna(0)

U, sigma, Vt = np.linalg.svd(ratings_matrix_svd, full_matrices=False)
sigma_diag = np.diag(sigma)

k = 10
predicted_ratings = np.dot(np.dot(U[:, :k], sigma_diag[:k, :k]), Vt[:k, :])

true_ratings = ratings_matrix_svd.values.flatten()
predicted_ratings_flat = predicted_ratings.flatten()

non_zero_indices = true_ratings > 0
true_ratings_filtered = true_ratings[non_zero_indices]
predicted_ratings_filtered = predicted_ratings_flat[non_zero_indices]

mse = mean_squared_error(true_ratings_filtered, predicted_ratings_filtered)
rmse = np.sqrt(mse)

In [52]:
print(f"MSE: {mse}")
print(f"RMSE: {rmse}")

MSE: 0.46598235920995756
RMSE: 0.682629005543976


# Conclusion

<b>Performance Comparison</b>
<br>The performance comparison reveals that the Ordinary SVD model with zero-filling achieves a lower RMSE of 0.6826, indicating better predictive performance compared to the Stochastic Gradient Descent (SGD) model, which has a higher RMSE of 0.9578. This suggests that the predictions made by the SVD model are closer to the actual values.

<b>Interpretation of Results</b>
<br>The results indicate that the SVD model outperforms the SGD model in this instance. The lower RMSE associated with the SVD model demonstrates its ability to make more accurate predictions. This could be attributed to the SVD method's simplicity, which can be advantageous in situations where the dataset is small.

<b>Data Limitations</b>
<br>The dataset's small size may hinder complex models like SGD from generalizing effectively, resulting in poorer performance. Conversely, the simpler Ordinary SVD approach tends to be more robust when working with limited data points.

<b>Model Complexity</b>
<br>The complexity of the SGD model can lead to overfitting the training data or failing to discern meaningful patterns when the dataset is insufficient. In contrast, the zero-filling SVD method is more straightforward and may provide more stable results in scenarios where data is limited.